### Imports


In [19]:
from pprint import pprint
from tqdm.auto import tqdm
from haystack.nodes import QuestionGenerator, BM25Retriever, FARMReader
from haystack.document_stores import ElasticsearchDocumentStore
from haystack.pipelines import (
    QuestionGenerationPipeline,
    RetrieverQuestionGenerationPipeline,
    QuestionAnswerGenerationPipeline,
)
from haystack.utils import launch_es, print_questions, add_example_data
from haystack import Pipeline
from haystack.document_stores import InMemoryDocumentStore

### Logging configuration


In [20]:
import logging

logging.basicConfig(
    format="%(levelname)s - %(name)s -  %(message)s", level=logging.WARNING
)
logging.getLogger("haystack").setLevel(logging.INFO)

### Question Generator


In [43]:
document_store = InMemoryDocumentStore()
text1 = "There are many other types of Enabler stories, including: 1) Refactoring and Spikes, 2) Building or improving development/deployment infrastructure, 3) Running jobs that require human interaction, 4) Creating the required product or component configurations for different purposes, 5) Verification of system qualities. Enabler stories are demonstrated just like user stories, typically by showing the knowledge gained, artifacts produced, or the user interface, stub, or mock-up."
text2 = "User stories are the primary means of expressing needed functionality. They essentially replace the traditional requirements specification. In some cases, however, they serve as a means to explain and develop system behavior later recorded in specifications supporting compliance, suppliers, traceability, or other needs."
text3 = "The Agile Team focus on the user as the subject of interest and not the system, user stories are value and customer-centric. To support this, the recommended form of expression is the ‘user-voice form,’ as follows: As a (user role), I want to (activity) so that (business value) By using this format, the teams are guided to understand who is using the system, what they are doing with it, and why they are doing it. Applying the ‘user voice’ format routinely tends to increase the team’s domain competence; they come to better understand the real business needs of their user."

docs = [{"content": text1}, {"content": text2}, {"content": text3}]
document_store.write_documents(docs)

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0


In [ ]:
question_generator = QuestionGenerator()
question_generation_pipeline = QuestionGenerationPipeline(question_generator)
for idx, document in enumerate(document_store):
    print(
        f"\n * Generating questions for document {idx}: {document.content[:100]}...\n"
    )
    result = question_generation_pipeline.run(documents=[document])
    print_questions(result)

### Question and Answer Generator


In [34]:
document_store = InMemoryDocumentStore()
add_example_data(document_store, "data/")
pprint(document_store.get_document_count())

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.utils.getting_started -  Adding 8 number of files from local disk at data/.
INFO - haystack.utils.preprocessing -  Converting data/Epic.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_5.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_4.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_6.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_3.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_2.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_1.txt
INFO - haystack.utils.preprocessing -  Converting data/Iteration_Planning.txt
Preprocessing: 100%|██████████| 8/8 [00:00<00:00, 1723.75docs/s]

8


In [42]:
question_generator = QuestionGenerator()
reader = FARMReader("deepset/roberta-base-squad2")
question_answer_generation_pipeline = QuestionAnswerGenerationPipeline(
    question_generator, reader
)
for idx, document in enumerate(tqdm(document_store)):
    print(
        f"\n * Generating questions and answers for document {idx}: {document.content[:100]}...\n"
    )
    result = question_answer_generation_pipeline.run(documents=[document])
    print_questions(result)

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
Using sep_token, but it is not set yet.
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.modeling.model.language_model -   * LOADING MODEL: 'deepset/roberta-base-squad2' (Roberta)
INFO - haystack.modeling.model.language_model -  Auto-detected model language: english
INFO - haystack.modeling.model.language_model -  Loaded 'deepset/roberta-base-squad2' (Roberta model) from model hub.
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0


0it [00:00, ?it/s]


 * Generating questions and answers for document 0: There are many other types of Enabler stories, including: 1) Refactoring and Spikes, 2) Building or ...



Inferencing Samples: 100%|██████████| 1/1 [00:00<00:00,  1.04 Batches/s]



Generated pairs:
 - Q: What are two examples of Enabler stories?
      A: Refactoring and Spikes
 - Q: What is one example of a user story?
      A: Enabler
 - Q: How are user stories demonstrated?
      A: by showing the knowledge gained, artifacts produced, or the user interface, stub, or mock-up
 - Q: What are enabler stories similar to?
      A: user stories
 - Q: What is an example of an enabler story?
      A: Refactoring and Spikes

 * Generating questions and answers for document 1: User stories are the primary means of expressing needed functionality. They essentially replace the ...



Inferencing Samples: 100%|██████████| 1/1 [00:00<00:00,  1.62 Batches/s]



Generated pairs:
 - Q: What are the primary means of expressing needed functionality?
      A: User stories
 - Q: What replaces the traditional requirements specification?
      A: User stories
 - Q: User stories serve as a means to explain and develop what?
      A: system behavior

 * Generating questions and answers for document 2: The Agile Team focus on the user as the subject of interest and not the system, user stories are val...



Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.16s/ Batches]


Generated pairs:
 - Q: What does the Agile Team focus on?
      A: the user
 - Q: What is the recommended form of expression?
      A: user-voice form
 - Q: What format helps teams understand who is using the system, what they are doing with it, and why they are using it?
      A: user-voice form
 - Q: What does applying the ‘user voice’ format routinely tend to increase?
      A: the team’s domain competence
 - Q: What does routinely tend to increase the team’s domain competence?
      A: Applying the ‘user voice’ format
 - Q: They come to better understand what?
      A: real business needs of their user
